# 훈련량 예측

5장에서 저장한 평가 데이터를 바탕으로, 추천일 이전 기록을 이용해
추천일 당일부터 이후 6일까지의 7일 TSS 합계를 예측한다.

이 예측은 앞으로 기록될 훈련량을 대상으로 하며, 실제 피로·회복
상태나 개별 훈련 처방을 확정하지 않는다.


In [1]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

evaluation_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_recommendation_evaluation.csv"
)

evaluation_df = pd.read_csv(
    evaluation_path,
    parse_dates=["recommendation_date"],
)

evaluation_df.shape

(100, 18)

In [2]:
evaluation_df = (
    evaluation_df
    .sort_values("recommendation_date")
    .reset_index(drop=True)
)

evaluation_df["recommendation_date"].agg(
    ["min", "max"]
)

min   2009-03-21 00:00:00+00:00
max   2009-06-28 00:00:00+00:00
Name: recommendation_date, dtype: datetime64[us, UTC]

## 1. 입력 후보와 정답값 선택

입력 후보는 추천일 이전 기록으로 계산한 이전 7일·28일 TSS,
이전 7일 운동 시간·라이드 횟수, 전날까지의 연속 라이드 기록일과
연속 무기록일 수로 선택한다. 무기록일은 실제 휴식일로 단정하지 않는다.

정답값 `next_7_day_tss`는 추천일 당일부터 7일간 기록된 TSS 합계다.
이후 TSS 변화량·비율은 미래 기록을 포함하므로 입력에서 제외한다.
부하 라벨과 안내 문장도 이번 입력 후보에서는 제외한다.


In [3]:
feature_columns = [
    "previous_7_day_tss",
    "previous_28_day_tss",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

target_column = "next_7_day_tss"

prediction_df = evaluation_df[
    ["recommendation_date"]
    + feature_columns
    + [target_column]
].copy()

prediction_df.head().round(
    {
        "previous_7_day_tss": 2,
        "previous_28_day_tss": 2,
        "previous_7_day_workout_hours": 2,
        "next_7_day_tss": 2,
    }
)

,recommendation_date,previous_7_day_tss,previous_28_day_tss,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days,next_7_day_tss
0,2009-03-21 00:00:00+00:00,829.25,2506.27,11.19,8.0,1.0,0.0,435.28
1,2009-03-22 00:00:00+00:00,926.31,2663.95,12.62,8.0,2.0,0.0,391.33
2,2009-03-23 00:00:00+00:00,812.04,2417.65,10.73,6.0,3.0,0.0,458.54
3,2009-03-24 00:00:00+00:00,709.89,2417.65,9.16,5.0,0.0,1.0,630.81
4,2009-03-25 00:00:00+00:00,580.91,2417.65,7.71,4.0,0.0,2.0,630.81


In [4]:
prediction_df[
    feature_columns + [target_column]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   previous_7_day_tss                      100 non-null    float64
 1   previous_28_day_tss                     100 non-null    float64
 2   previous_7_day_workout_hours            100 non-null    float64
 3   previous_7_day_ride_count               100 non-null    float64
 4   previous_day_recorded_ride_streak_days  100 non-null    float64
 5   previous_day_no_record_streak_days      100 non-null    float64
 6   next_7_day_tss                          100 non-null    float64
dtypes: float64(7)
memory usage: 5.6 KB


In [5]:
prediction_df[
    feature_columns + [target_column]
].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
previous_7_day_tss,100.0,705.09,143.53,362.95,606.98,716.42,821.39,1012.40
previous_28_day_tss,100.0,2800.49,270.56,2213.47,2636.21,2842.41,3017.11,3309.16
previous_7_day_workout_hours,100.0,9.88,2.24,5.16,7.97,9.78,11.58,14.86
previous_7_day_ride_count,100.0,5.43,1.39,3.00,4.00,5.00,6.00,8.00
previous_day_recorded_ride_streak_days,100.0,2.04,2.49,0.00,0.00,1.00,3.00,12.00
previous_day_no_record_streak_days,100.0,0.51,0.86,0.00,0.00,0.00,1.00,4.00
next_7_day_tss,100.0,687.13,155.72,362.95,581.91,706.32,802.62,1012.40


In [6]:
prediction_df[
    "recommendation_date"
].diff().value_counts()

recommendation_date
1 days    99
Name: count, dtype: int64

In [7]:
evaluation_start_date = prediction_df[
    "recommendation_date"
].iloc[80]

training_cutoff_date = (
    evaluation_start_date
    - pd.Timedelta(days=6)
)

train_df = prediction_df.loc[
    prediction_df["recommendation_date"]
    < training_cutoff_date
].copy()

test_df = prediction_df.loc[
    prediction_df["recommendation_date"]
    >= evaluation_start_date
].copy()

train_df.shape, test_df.shape

((74, 8), (20, 8))

In [8]:
baseline_predictions = test_df[
    "previous_7_day_tss"
]

baseline_absolute_errors = (
    test_df[target_column]
    - baseline_predictions
).abs()

baseline_mae = baseline_absolute_errors.mean()

baseline_mae

np.float64(164.30677149999997)

### 예측 데이터 점검 및 기준선 평가 결과

- 5장에서 저장한 100행 18열의 평가 데이터를 불러와, 추천일과 입력 후보 여섯 개, 정답값으로 구성된 100행 8열의 예측용 표를 만들었다.
- 입력 후보는 이전 7일·28일 TSS, 이전 7일 운동 시간·라이드 횟수, 전날까지의 연속 라이드 기록일·연속 무기록일 수로 선택했다. 정답값은 추천일 당일부터 이후 6일까지의 TSS 합계인 `next_7_day_tss`다.
- 입력 후보와 정답값은 모두 숫자형 `float64`였으며, 각 열의 100개 값에 결측값이 없었다. 요약 통계에서 음수나 제거 근거가 있는 명백한 비정상 값은 발견하지 않았다. 연속 라이드 기록일의 최댓값 12일은 기존 분석에서 확인한 기록이므로 유지했다.
- 추천일은 2009년 3월 21일부터 6월 28일까지 하루 간격으로 이어졌다. 앞쪽 74일을 학습 기간으로, 마지막 20일을 평가 기간으로 나누었다. 학습용 7일 정답 기간이 평가 시작일에 걸치지 않도록 평가 직전 6일은 학습에서 제외했다.
- 이후 7일 TSS가 이전 7일 TSS와 같다고 예측하는 단순 예측 기준선(baseline)을 만들었다. 평가 기간 20일의 평균 절대오차(MAE)는 약 164.31 TSS였다.
- 이 기준선은 같은 평가 기간에서 선형회귀 모델의 MAE와 비교하는 기준으로 사용한다.

이 기준선은 앞으로 기록될 훈련량의 예측 성능을 비교하기 위한 것이다. 실제 피로·회복 상태나 개별 훈련 처방의 적절성을 평가하지 않으며, 무기록일을 실제 휴식일로 단정하지 않는다.

In [9]:
from sklearn.linear_model import LinearRegression

# 선형회귀 모델
model = LinearRegression()

# 모델 학습
model.fit(
    train_df[feature_columns],  # 훈련 데이터셋 입력값
    train_df[target_column],  # 훈련 데이터셋 정닶값
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](6,)","[ 0.6 , 0.22,-77.84, 18.63, 3.09, 16.86]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](6,)","['previous_7_day_tss','previous_28_day_tss','previous_7_day_workout_hours', 'previous_7_day_ride_count','previous_day_recorded_ride_streak_days', 'previous_day_no_record_streak_days']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,354.4
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(6)


In [10]:
coefficient_series = pd.Series(
    model.coef_,  # 모델이 학습한 계수
    index=feature_columns,
)

print("모델이 학습한 계수:")
print(coefficient_series)

print("모델의 절편:", model.intercept_)

모델이 학습한 계수:
previous_7_day_tss                         0.598573
previous_28_day_tss                        0.220830
previous_7_day_workout_hours             -77.836626
previous_7_day_ride_count                 18.629305
previous_day_recorded_ride_streak_days     3.090415
previous_day_no_record_streak_days        16.862714
dtype: float64
모델의 절편: 354.4057162311606


In [11]:
# 학습 모델로 예측
test_predictions = model.predict(
    test_df[feature_columns],  # 테스트 데이터셋 입력값
)

print("테스트 데이터의 예측값:")
print(test_predictions)

테스트 데이터의 예측값:
[872.44810047 852.00743172 856.653299   826.97070642 876.64528092
 881.99170113 806.89605583 793.57899564 810.44170996 793.10022949
 789.27624441 846.89031388 769.79276788 813.2368632  794.42114028
 796.00956523 800.05277788 737.22103515 754.33238081 776.49182756]


In [12]:
test_absolute_errors = (
    test_df[target_column]
    - test_predictions
).abs()

test_mae = test_absolute_errors.mean()

print("Baseline MAE:", baseline_mae)
print("선형회귀 테스트 MAE:", test_mae)

Baseline MAE: 164.30677149999997
선형회귀 테스트 MAE: 268.75926734401673


In [13]:
# 훈련 데이터셋으로 학습한 모델에 다시 훈련 데이터셋을 입력하여 MAE 확인
train_predictions = model.predict(
    train_df[feature_columns],
)

train_absolute_errors = (
    train_df[target_column]
    - train_predictions
).abs()

train_mae = train_absolute_errors.mean()

print("선형회귀 훈련 MAE:", train_mae)
print("선형회귀 테스트 MAE:", test_mae)

선형회귀 훈련 MAE: 97.35080989905981
선형회귀 테스트 MAE: 268.75926734401673


### 선형회귀 모델 학습 및 평가 결과

- 입력 후보 여섯 개와 이후 7일 TSS를 사용해 훈련 기간 74일에서 선형회귀 모델을 학습했다. 모델이 학습한 계수 여섯 개와 절편을 확인했으며, 절편은 약 354.41이었다.
- 학습한 모델에 테스트 기간 20일의 입력값을 전달해 이후 7일 TSS를 예측했다. 예측에는 정답값을 전달하지 않고, 예측 후 실제 정답값과 비교해 평균 절대오차(MAE)를 계산했다.
- 선형회귀의 테스트 MAE는 약 268.76 TSS로, 같은 테스트 기간의 기준선 MAE 약 164.31 TSS보다 약 104.45 TSS 컸다. 이번 테스트 기간에서는 선형회귀가 단순 예측보다 오차를 줄이지 못했다.
- 같은 모델에 훈련 기간의 입력값을 다시 전달해 예측한 결과, 훈련 MAE는 약 97.35 TSS였다. 이 과정은 모델을 다시 학습하는 것이 아니라 이미 학습한 모델이 훈련 데이터에 얼마나 잘 맞는지 확인한 것이다.
- 훈련 데이터보다 테스트 데이터에서 오차가 크게 증가했다. 과적합 가능성이나 기간별 훈련 패턴의 차이를 검토할 수 있지만, 현재 결과만으로 원인을 확정하지 않는다.
- 다음 단계에서는 테스트 날짜별 실제 TSS와 예측 TSS를 비교해, 지속적으로 높거나 낮게 예측하는지 또는 특정 날짜에 오차가 집중되는지 확인한다.

이 평가는 기록될 훈련량의 예측 성능을 확인한 것이다. 실제 피로·회복 상태나 개별 훈련 처방의 적절성을 확정하지 않는다.


In [14]:
test_result_df = test_df[
    ["recommendation_date", target_column]
].copy()

test_result_df["predicted_tss"] = test_predictions

# 정답값인 이후 7일 TSS와 예측값을 비교
test_result_df.head().round(
    {
        "next_7_day_tss": 2,
        "predicted_tss": 2,
    }
)

,recommendation_date,next_7_day_tss,predicted_tss
80,2009-06-09 00:00:00+00:00,716.71,872.45
81,2009-06-10 00:00:00+00:00,716.71,852.01
82,2009-06-11 00:00:00+00:00,828.17,856.65
83,2009-06-12 00:00:00+00:00,707.51,826.97
84,2009-06-13 00:00:00+00:00,560.77,876.65


In [15]:
# 예측값에서 정답값을 뺀 오차 열 생성
test_result_df["prediction_error"] = (
    test_result_df["predicted_tss"]
    - test_result_df[target_column]
)

test_result_df[
    "prediction_error"
].describe().round(2)

count     20.00
mean     268.76
std      127.52
min       28.48
25%      156.74
50%      288.80
75%      365.79
max      445.17
Name: prediction_error, dtype: float64

In [16]:
# 훈련 데이터셋과 테스트 데이터셋의 정닶값을 비교
target_summary_df = pd.DataFrame(
    {
        "train": train_df[target_column].describe(),
        "test": test_df[target_column].describe(),
    }
)

target_summary_df.round(2)

,train,test
count,74.00,20.00
mean,723.29,543.66
std,140.46,147.04
min,391.33,362.95
25%,630.81,409.67
50%,720.21,547.01
75%,835.58,709.81
max,1012.40,828.17


In [17]:
# 학습 데이터셋과 테스트 데이터셋의 각 feature 평균 비교
feature_mean_df = pd.DataFrame(
    {
        "train": train_df[feature_columns].mean(),
        "test": test_df[feature_columns].mean(),
    }
)

feature_mean_df.round(2)

,train,test
previous_7_day_tss,715.17,635.26
previous_28_day_tss,2735.70,2955.74
previous_7_day_workout_hours,10.04,8.72
previous_7_day_ride_count,5.51,4.75
previous_day_recorded_ride_streak_days,2.43,0.85
previous_day_no_record_streak_days,0.46,0.75


In [18]:
feature_change_df = feature_mean_df.copy()

# 각 feature 평균 값에 대해 테스트 데이터셋 - 훈련 데이터셋으로 평균 변화 확인
feature_change_df["mean_change"] = (
    feature_change_df["test"]
    - feature_change_df["train"]
)

# 평균 변화에 각 feature의 계수를 곱하여 예측 방향을 확인
feature_change_df["prediction_contribution"] = (
    feature_change_df["mean_change"]
    * coefficient_series
)

feature_change_df.round(2)

,train,test,mean_change,prediction_contribution
previous_7_day_tss,715.17,635.26,-79.90,-47.83
previous_28_day_tss,2735.70,2955.74,220.04,48.59
previous_7_day_workout_hours,10.04,8.72,-1.32,102.58
previous_7_day_ride_count,5.51,4.75,-0.76,-14.22
previous_day_recorded_ride_streak_days,2.43,0.85,-1.58,-4.89
previous_day_no_record_streak_days,0.46,0.75,0.29,4.90


### 테스트 기간의 과대예측 확인 결과

- 테스트 날짜별 실제 이후 7일 TSS와 선형회귀 예측값을 같은 표로 비교하고, 예측값에서 실제 값을 뺀 방향성 있는 오차를 계산했다.
- 테스트 20일 모두 실제보다 높게 예측했다. 오차의 최솟값은 약 +28.48 TSS, 최댓값은 약 +445.17 TSS였다. 모든 오차가 양수여서 평균 오차와 MAE는 약 268.76 TSS로 같았다.
- 실제 이후 7일 TSS의 평균은 훈련 기간 723.29에서 테스트 기간 543.66으로 약 179.63 낮아졌다. 중앙값도 720.21에서 547.01로 낮아졌다.
- 테스트 기간에는 이전 7일 TSS, 운동 시간, 라이드 횟수의 평균이 낮아졌지만, 이전 28일 TSS의 평균은 2735.70에서 2955.74로 높아졌다. 최근 운동량은 감소했지만 장기 누적 부하는 높은 모습이었다.
- 각 입력값의 평균 변화에 모델 계수를 곱해 두 기간의 평균 예측값 차이에 대한 기여를 확인했다. 이전 7일 TSS 감소는 약 -47.83 TSS, 이전 28일 TSS 증가는 약 +48.59 TSS, 이전 7일 운동 시간 감소는 약 +102.58 TSS로 작용했다.
- 여섯 입력값의 기여를 합하면 약 +89.13 TSS였다. 같은 모델의 평균 예측은 테스트 기간에서 더 높아진 반면, 실제 이후 훈련량의 평균은 낮아졌다. 절편은 두 기간에서 같으므로 평균 예측값 차이에서는 상쇄된다.
- 이 기여는 모델의 계산식에서 두 기간의 평균 예측값 차이를 분해한 것이며, 개별 feature가 만든 예측 오차나 인과 효과를 뜻하지 않는다. 과대예측의 패턴과 계산 방향은 확인했지만, 모델이 이런 계수를 학습한 원인은 아직 확정하지 않았다.
- 다음 단계에서는 훈련 데이터의 TSS와 운동 시간 관계를 확인해, 서로 연관된 입력값을 함께 사용한 영향이 있는지 살펴본다.


In [19]:
train_df[
    [
        "previous_7_day_tss",
        "previous_7_day_workout_hours",
    ]
].corr().round(2)

,previous_7_day_tss,previous_7_day_workout_hours
previous_7_day_tss,1.00,0.94
previous_7_day_workout_hours,0.94,1.00


In [20]:
train_df[
    [
        "previous_7_day_tss",
        "previous_7_day_workout_hours",
        target_column,
    ]
].corr().round(2)

,previous_7_day_tss,previous_7_day_workout_hours,next_7_day_tss
previous_7_day_tss,1.00,0.94,-0.23
previous_7_day_workout_hours,0.94,1.00,-0.32
next_7_day_tss,-0.23,-0.32,1.00


### 훈련 데이터의 TSS와 운동 시간 상관관계 확인 결과

- 훈련 기간 74일에서 이전 7일 TSS와 이전 7일 운동 시간의 피어슨 상관계수를 확인했다. 상관계수는 약 0.94로, 두 입력값은 강한 양의 선형 관계를 보였다.
- 이후 7일 TSS와의 상관계수는 이전 7일 TSS가 약 -0.23, 이전 7일 운동 시간이 약 -0.32였다. 이 훈련 기간에서는 최근 운동량이 많을수록 이후 운동량이 낮아지는 경향이 있었지만, 두 변수만의 선형 관계는 입력값끼리의 관계보다 약했다.
- 이전 7일 TSS는 정답값과의 단순 상관이 음수였지만, 선형회귀 모델의 계수는 약 +0.60이었다. 운동 시간은 단순 상관과 모델 계수가 모두 음수였다.
- 단순 상관은 다른 입력값을 고정하지 않은 두 변수의 관계이며, 모델 계수는 다른 입력값이 같을 때의 예측 변화를 나타낸다. 서로 강하게 연결된 입력값을 함께 사용했으므로, 개별 계수를 일반적인 훈련 원리나 인과 효과로 해석하지 않는다.
- 강한 입력값 간 상관은 계수 해석과 안정성을 검토할 단서지만, 이번 상관분석만으로 음수 계수나 테스트 과대예측의 원인을 확정하지 않는다. 모델이나 입력값은 아직 변경하지 않았다.
- 다음 단계에서는 훈련 데이터에서 다른 입력값의 관계도 살펴보고, 입력 구성이 계수에 미치는 영향을 확인할 방법을 정한다. 테스트 결과를 반복적으로 참고해 모델을 바꾸면 평가의 독립성이 약해지므로, 모델 변경을 비교할 때는 훈련 기간 안에서 시간순 검증을 준비한다.


In [21]:
# feature 간의 상관계수를 확인
feature_correlation_df = train_df[
    feature_columns
].corr()

feature_correlation_df.round(2)

,previous_7_day_tss,previous_28_day_tss,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days
previous_7_day_tss,1.00,0.52,0.94,0.80,0.27,-0.21
previous_28_day_tss,0.52,1.00,0.62,0.53,0.31,-0.25
previous_7_day_workout_hours,0.94,0.62,1.00,0.73,0.30,-0.17
previous_7_day_ride_count,0.80,0.53,0.73,1.00,0.48,-0.32
previous_day_recorded_ride_streak_days,0.27,0.31,0.30,0.48,1.00,-0.48
previous_day_no_record_streak_days,-0.21,-0.25,-0.17,-0.32,-0.48,1.00


In [22]:
# 이전 7일 TSS와 0.94의 높은 상관계수를 보인 운동 시간을 feature에서 제거, 새로 학습한 모델의 계수 변화를 확인
reduced_feature_columns = feature_columns.copy()
reduced_feature_columns.remove(
    "previous_7_day_workout_hours"
)

reduced_model = LinearRegression()

reduced_model.fit(
    train_df[reduced_feature_columns],
    train_df[target_column],
)

coefficient_comparison_df = pd.DataFrame(
    {
        "original": coefficient_series,
        "without_workout_hours": pd.Series(
            reduced_model.coef_,
            index=reduced_feature_columns,
        ),
    }
)

coefficient_comparison_df.round(2)

,original,without_workout_hours
previous_28_day_tss,0.22,0.09
previous_7_day_ride_count,18.63,34.77
previous_7_day_tss,0.60,-0.58
previous_7_day_workout_hours,-77.84,NaN
previous_day_no_record_streak_days,16.86,0.70
previous_day_recorded_ride_streak_days,3.09,-2.61


### 입력 간 상관관계와 입력 변경에 따른 계수 변화 확인 결과

- 훈련 데이터의 입력 여섯 개 사이의 상관관계를 확인했다. 이전 7일 TSS와 운동 시간은 약 0.94, TSS와 라이드 횟수는 약 0.80, 운동 시간과 라이드 횟수는 약 0.73의 양의 상관을 보였다. 최근 운동량을 나타내는 입력들에 겹치는 정보가 있을 가능성을 확인했다.
- 높은 상관은 함께 움직이는 선형 관계를 뜻하며, 서로의 인과 효과나 완전히 같은 정보를 담고 있다는 것을 증명하지 않는다. 상관계수만으로 입력을 제거하기로 결정하지 않았다.
- 기존 모델을 유지하고, 이전 7일 운동 시간만 제외한 별도의 선형회귀 모델을 같은 훈련 기간 74일에서 학습했다. 이 실험은 예측 성능 개선보다 입력 구성에 따른 계수의 민감도를 확인하기 위한 것이다.
- 이전 7일 TSS 계수는 약 +0.60에서 -0.58로 부호가 바뀌었다. 이전 28일 TSS 계수는 약 +0.22에서 +0.09로, 라이드 횟수 계수는 약 +18.63에서 +34.77로 바뀌었다.
- 입력을 제외한 뒤에는 남은 계수와 절편을 모두 다시 학습한다. 서로 연결된 입력들이 역할을 나누거나 보완할 수 있으므로, 정보가 겹치는 입력 하나를 제외해도 다른 계수가 크게 변할 수 있다.
- 이번 계수 변화는 입력 구성에 민감하다는 단서다. 개별 계수를 고정된 훈련 원리로 해석하지 않으며, 계수 변화만으로 정보 중복이나 과대예측의 원인을 확정하지 않는다. 계수 변화가 작더라도 입력의 독립성을 뜻하지 않는다.
- 계수 변화와 예측값 변화는 구분한다. 운동 시간을 제외한 모델의 예측 성능과 과대예측 개선 여부는 아직 평가하지 않았다. 이후 모델 구성을 비교할 때는 훈련 기간 안에서 시간순 검증을 준비해 검증 오차를 함께 확인한다.


In [23]:
# feature 구성에 따른 모델 비교를 위해 검증 데이터셋으로 14개의 샘플을 분할
validation_start_date = train_df[
    "recommendation_date"
].iloc[-14]

# 학습 데이터셋의 정답이 되는 이후 7일 TSS가 검증 기간의 기록과 겹치지 않도록 6일 제외
fit_cutoff_date = (
    validation_start_date
    - pd.Timedelta(days=6)
)

# 학습 데이터셋
fit_df = train_df.loc[
    train_df["recommendation_date"]
    < fit_cutoff_date
].copy()

# 검증 데이터셋
validation_df = train_df.loc[
    train_df["recommendation_date"]
    >= validation_start_date
].copy()

print("학습용 표 크기:", fit_df.shape)
print("검증용 표 크기:", validation_df.shape)

학습용 표 크기: (54, 8)
검증용 표 크기: (14, 8)


In [24]:
# 6개의 feature로 구성된 데이터셋으로 학습
validation_model = LinearRegression()

validation_model.fit(
    fit_df[feature_columns],
    fit_df[target_column],
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](6,)","[ 1.53, 0.33,-122.45, -35.77, -37.74, -26.55]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](6,)","['previous_7_day_tss','previous_28_day_tss','previous_7_day_workout_hours', 'previous_7_day_ride_count','previous_day_recorded_ride_streak_days', 'previous_day_no_record_streak_days']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,207
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(6)


In [26]:
validation_predictions = validation_model.predict(
    validation_df[feature_columns],
)

validation_absolute_errors = (
    validation_df[target_column]
    - validation_predictions
).abs()

validation_mae = validation_absolute_errors.mean()

print("입력 6개 모델의 검증 MAE:", validation_mae)

입력 6개 모델의 검증 MAE: 289.0314875410439


In [34]:
# 운동 시간을 제외한 5개의 feature로 구성된 데이터셋으로 학습
reduced_validation_model = LinearRegression()

reduced_validation_model.fit(
    fit_df[reduced_feature_columns],
    fit_df[target_column],
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[ -0.31, 0. , -0.57,-39.9 ,-43.16]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['previous_7_day_tss','previous_28_day_tss','previous_7_day_ride_count', 'previous_day_recorded_ride_streak_days', 'previous_day_no_record_streak_days']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1002
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)


In [36]:
reduced_validation_predictions = (
    reduced_validation_model.predict(
        validation_df[reduced_feature_columns],
    )
)

reduced_validation_absolute_errors = (
    validation_df[target_column]
    - reduced_validation_predictions
).abs()

reduced_validation_mae = (
    reduced_validation_absolute_errors.mean()
)

print("입력 5개 모델의 검증 MAE:", reduced_validation_mae)

입력 5개 모델의 검증 MAE: 276.82640990132575


In [37]:
# 기존 baseline 예측과 비교
validation_baseline_predictions = validation_df[
    "previous_7_day_tss"
]

validation_baseline_absolute_errors = (
    validation_df[target_column]
    - validation_baseline_predictions
).abs()

validation_baseline_mae = (
    validation_baseline_absolute_errors.mean()
)

print("검증 Baseline MAE:", validation_baseline_mae)
print("입력 6개 모델의 검증 MAE:", validation_mae)
print("입력 5개 모델의 검증 MAE:", reduced_validation_mae)

검증 Baseline MAE: 151.20471142857136
입력 6개 모델의 검증 MAE: 289.0314875410439
입력 5개 모델의 검증 MAE: 276.82640990132575


### 시간순 검증 기간의 모델 및 기준선 비교 결과

- 입력 여섯 개를 사용하는 선형회귀와 운동 시간을 제외한 입력 다섯 개의 선형회귀를 학습용 54일에서 각각 새로 학습했다. 기존 훈련 74일 전체를 학습한 모델은 이번 검증에 사용하지 않았다.
- 두 모델을 같은 검증 기간 14일에서 평가했다. 입력 여섯 개 모델의 MAE는 약 289.03 TSS, 입력 다섯 개 모델의 MAE는 약 276.83 TSS였다.
- 운동 시간을 제외한 모델의 검증 MAE는 약 12.21 TSS, 4.2% 낮아졌다. 이번 검증 기간에서는 소폭 개선되었지만, 모든 기간에서 운동 시간을 제외하는 것이 유리하다고 확정하지 않는다.
- 같은 검증 기간에서 이후 7일 TSS가 이전 7일 TSS와 같다고 예측하는 기준선의 MAE는 약 151.20 TSS였다. 기준선의 오차가 두 선형회귀 모델보다 작았다.
- 운동 시간 제거만으로는 기준선보다 나은 예측 성능을 확보하지 못했다. 입력 간 높은 상관과 계수 변화만으로 예측 성능 개선을 판단하지 않고, 실제 검증 오차를 함께 확인했다.
- 기존 테스트 기간과 이번 검증 기간은 날짜가 다르므로 MAE 수치의 차이를 모델 개선으로 직접 해석하지 않는다. 이번 비교는 한 번의 시간순 분할에 대한 결과이며, 선형회귀 전체의 적합성이나 오차 원인을 확정하지 않는다.
- 다음 단계에서는 검증 기간의 실제 값과 예측값, 학습용 기간과 검증 기간의 훈련량 차이를 살펴보며 이번 검증 오차의 패턴을 확인한다. 새로운 모델 변경은 아직 결정하지 않았다.
